In [28]:
import pandas as pd
import numpy as np


import xgboost as xgb

from sklearn.metrics import classification_report, roc_auc_score

import matplotlib.pyplot as plt

In [32]:
print(xgb.__version__)

3.2.0


In [15]:
df = pd.read_csv("../data/processed/features_2023.csv")

```
unlike logistic regression, XGBoost doesn't care about feature scale, multicollinearity, or even raw categorical text in some cases, so we use the .csv file rather than the scaled version
```

In [34]:
print(f"Shape of the dataset: {df.shape}")

Shape of the dataset: (11089, 34)


In [40]:
# Split races 

test_races = ["Singapore", "Monza"]

train_df = df[~df["RaceName"].isin(test_races)]
test_df = df[df["RaceName"].isin(test_races)]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (9079, 34)
Test shape: (2010, 34)


In [48]:
# Seperate Features(X) from target (y)

exclude_cols = [
    "Pitted", "Driver", "RaceName", "Compound",
    "IsAccurate", "FastF1Generated", "IsPersonalBest"
]

feature_col = [col for col in train_df.columns if col not in exclude_cols]

X_train = train_df[feature_col]
y_train = train_df["Pitted"]

X_test =test_df[feature_col]
y_test = test_df["Pitted"]


print("Feature columns", feature_col, "\n\n")
print(f"X_train shape: {X_train.shape}\n\n")
print(f"y_train shape: {y_train.shape}")

Feature columns ['Time', 'LapTime', 'LapNumber', 'Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'LapStartTime', 'TrackStatus', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace'] 


X_train shape: (9079, 27)


y_train shape: (9079,)


 ---

```
Time, LapStartTime, Sector1/2/3SessionTime, LapNumber
→ all measuring essentially "race progress" in different units
→ including all of them doesn't help, just adds redundant noise 
  and makes the model slightly slower without real benefit

```

In [52]:
redundant_time_cols = [
    "Time", "LapStartTime",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapNumber",  # superseded by RacePctComplete
    "LapTime"   # superseded by LapTimeRolling3

]

feature_col = [col for col in feature_col if col not in redundant_time_cols]

X_train =train_df[feature_col]
X_test = test_df[feature_col]

print("Cleaned feature columns:", feature_col)
print(f"\nX_train shape: {X_train.shape}")

Cleaned feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'TrackStatus', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

X_train shape: (9079, 20)


In [54]:
# Training the model 
# Handle class imbalance (33 : 1)
# scale_pos_weight tells XGBoost how muh more to weight the rare class

scale_pos_weight = (y_train == 0).sum() /(y_train ==1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

scale_pos_weight: 32.1


In [56]:
model_xgb = xgb.XGBClassifier(
    n_estimators = 300, # no of decision trees to build.
    max_depth = 6, # how deep each individual tree can grow
    learning_rate = 0.05,  # how much each new tree corrects the mistakes of previous trees 
    subsample = 0.8,
    colsample_bytree = 0.8, # each tree only see 80% of the rows+ cols (randomly choosen) this adds randomness helps prevent overfitting
    scale_pos_weight = scale_pos_weight,
    eval_metric = "auc",
    random_state = 42,
    n_jobs = -1

)
model_xgb.fit(X_train, y_train)
print("Model trained")

Model trained


In [60]:
# Predictions

# get predictions on the unseen test races
y_pred = model_xgb.predict(X_test)
y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]

print("Logistic Regression Performance (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {auc:.4f}")


Logistic Regression Performance (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1959
           1       1.00      1.00      1.00        51

    accuracy                           1.00      2010
   macro avg       1.00      1.00      1.00      2010
weighted avg       1.00      1.00      1.00      2010

ROC-AUC: 1.0000
